In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        # 🔹 BLOQUE 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # 🔹 BLOQUE 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # 🔹 BLOQUE 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # 🔹 POOLING
        self.pool = nn.MaxPool2d(2, 2)

        # ⭐ CLAVE: evita depender del tamaño de entrada
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))

        # 🔹 CAPAS FULLY CONNECTED
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, num_classes)

        # 🔹 DROPOUT (opcional pero recomendable)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):

        # BLOQUE 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # BLOQUE 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # BLOQUE 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # ⭐ Reduce a tamaño fijo (7x7)
        x = self.adaptive_pool(x)

        # Aplanar
        x = x.view(x.size(0), -1)

        # Fully Connected
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
import os
import cv2

# 🔹 CONFIGURACIÓN
INPUT_IMAGES = "css-data/train/images"
INPUT_LABELS = "css-data/train/labels"
OUTPUT_DIR = "css-data/train/clase"

CLASSES = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']

def yolo_to_bbox(x_center, y_center, width, height, img_w, img_h):
    x_center *= img_w
    y_center *= img_h
    width *= img_w
    height *= img_h

    x1 = int(x_center - width / 2)
    y1 = int(y_center - height / 2)
    x2 = int(x_center + width / 2)
    y2 = int(y_center + height / 2)

    return x1, y1, x2, y2


def process_split():
    img_dir = INPUT_IMAGES
    label_dir = INPUT_LABELS
    output_split = OUTPUT_DIR

    os.makedirs(output_split, exist_ok=True)

    count = 0

    for file in os.listdir(img_dir):
        if not file.endswith((".jpg", ".png", ".jpeg")):
            continue

        img_path = os.path.join(img_dir, file)
        label_path = os.path.join(label_dir, file.replace(".jpg", ".txt").replace(".png", ".txt"))

        if not os.path.exists(label_path):
            continue

        image = cv2.imread(img_path)
        if image is None:
            continue

        h, w, _ = image.shape

        with open(label_path, "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x, y, bw, bh = map(float, parts[1:])

            x1, y1, x2, y2 = yolo_to_bbox(x, y, bw, bh, w, h)

            # 🔹 Asegurar límites válidos
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            crop = image[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            class_name = CLASSES[class_id]
            class_dir = os.path.join(output_split, class_name)
            os.makedirs(class_dir, exist_ok=True)

            output_path = os.path.join(class_dir, f"{file[:-4]}_{i}.jpg")

            cv2.imwrite(output_path, crop)
            count += 1

    print(f"{split}: {count} imágenes generadas")


def main():
    process_split()


if __name__ == "__main__":
    main()

FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: 'css-data/train/images\\train'